# Overview

This notebook writes a ton of plots. Simple ones that compare all columns to the response to detect relationships.

In [43]:
import polars as pl
import plotly.express as px
import polars.selectors as cs
import plotly.figure_factory as ff

data_directory = "../../data/energy"
filename = "s_energy.parquet"
df = pl.read_parquet(f"{data_directory}/{filename}")
# Filter for just 2024 to examine relationships
df = (
    df
    .filter(
        (pl.col("Year") == 2024) &
        (pl.col("EUI_WN").is_not_null())
    )
    .with_columns((pl.col("Year") - pl.col("YearBuilt")).alias("BuildingAge"))
    .drop(["YearBuilt", "Year", "Years", "NumYears", "EUI", "Year2025"])
)

In [44]:
df.glimpse()

Rows: 3473
Columns: 10
$ ID                   <i64> 1, 2, 3, 5, 8, 9, 10, 11, 12, 15
$ BuildingName         <str> 'MAYFLOWER PARK HOTEL', 'PARAMOUNT HOTEL', 'WESTIN HOTEL (Parent Building)', 'HOTEL MAX', 'WARWICK SEATTLE HOTEL', 'WEST PRECINCT (SEATTLE POLICE)', 'CAMLIN WORLDMARK HOTEL', 'PARAMOUNT THEATRE', 'COURTYARD BY MARRIOTT - PIONEER SQ', 'HOTEL MONACO'
$ BuildingType         <str> 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential'
$ Neighborhood         <str> 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN'
$ NumberofFloors       <i64> 12, 11, 41, 10, 18, 2, 11, 8, 15, 11
$ NumberofBuildings    <i64> 1, 1, 3, 1, 1, 1, 1, 1, 1, 1
$ PropertyGFABuildings <i64> 88434, 88502, 759392, 61320, 113580, 60090, 83008, 102761, 163984, 133884
$ EPAPropertyType      <str> 'Hotel', 'Hotel', 'Hotel'

In [28]:
numerical_columns = df.select(cs.numeric()).columns
numerical_columns

['ID',
 'YearBuilt',
 'NumberofFloors',
 'NumberofBuildings',
 'PropertyGFABuildings',
 'EUI_WN']

In [25]:
response = "EUI_WN"
columns = df.drop("ID", "BuildingName", response).columns
columns

['BuildingType',
 'Neighborhood',
 'YearBuilt',
 'NumberofFloors',
 'NumberofBuildings',
 'PropertyGFABuildings',
 'EPAPropertyType']

In [40]:
fig = px.box(
    df,
    y="BuildingType",
    x="EUI_WN",
    color="BuildingType"
)
fig.show()

In [48]:
boolean_columns = df.select(cs.boolean()).columns
categorical_columns = df.select(cs.string())
numerical_columns = df.select(cs.numeric()).columns
response = "EUI_WN"
columns = df.drop("ID", "BuildingName", response).columns
write = True
log_scale = True
for column in columns:

    if column in boolean_columns:
        fig = px.density_heatmap(
            df,
            x=response,
            y=column,
            text_auto=True,
            color_continuous_scale="Greys",
            title=f"Response breakdown for {column}"
        )

    elif column in categorical_columns:
        fig = px.box(
            df,
            y=column,
            x=response,
            color=column,
            log_x=log_scale,
            title=f"Response breakdown for {column}"
        )

    else:
        fig = px.scatter(
            df,
            x=response,
            y=column,
            log_x=log_scale,
            title=f"Response breakdown for {column}"
        )

    if write:
        fig.write_image(f"../../images/energy/{column}_vs_response.png")
    fig.show()